# Flower-visit aggregation pipeline (filtered, deceleration-gated)

Sweeps every `<date>_system_<sys>` folder in `data/flight_data/`, runs the
within-track + cross-track flower-visit detector, then applies quality
filters and emits aggregated indicators.

### The two flower-visit scenarios

1. **within-track** — a single track contains a contiguous stretch
   `≥ MIN_VISIT_S` where the bee's 3D displacement over a 1-second
   window is `< DISP_THR_M` (default 5 cm).  The bee is parked on a
   flower; no velocity check is needed because the displacement
   criterion already implies a speed `< 0.05 m/s`.
2. **cross-track** — track A's last (x,y,z) is within `LINK_DIST_M` of
   track B's first (x,y,z), and B starts between `MIN_VISIT_S` and
   `LINK_DT_MAX_S` after A ends.  PATS lost the bee briefly while it
   sat on the flower, then re-acquired it.  Because we don't observe
   the bee during the gap, we require **track A to be decelerating to
   `< DECEL_END_THR_MS` m/s in its last `VEL_WINDOW` frames** — a
   freshly-landed bee should be slowing down, not still cruising.

### Filters applied after detection

| filter | rule | rationale |
|---|---|---|
| `hive` | 3D distance from hive `< HIVE_RADIUS_M` | bee at the hive entrance, not on a flower |
| `post` | `|x − hive_x| < POST_HALF_WIDTH_M` **and** `|y − hive_y| < POST_HALF_WIDTH_M` | bee behind the 15×15 cm hive-mount post (PATS loses + reacquires → looks like a visit) |
| `fov_edge` | within `EDGE_BUFFER_M` of any axis's 1st/99th percentile boundary | bee flying out of (or into) PATS camera FOV — not a landing |
| `end_velocity` | (cross-track only) track A is not decelerating (slope ≥ 0) OR end-mean speed ≥ `DECEL_END_THR_MS` | bee was still flying when A ended → cross-track pair is a tracking dropout artefact |

Rejected visits are kept in `flower_visits_rejected.csv` for audit.

### Outputs

- `data/multi_day/flower_visits.csv` — accepted visits (one row per visit)
- `data/multi_day/flower_visits_rejected.csv` — rejected visits with `filter_reason`
- `data/multi_day/flower_visit_summary.csv` — per (date, system_id) indicators


### Post geometry (corrected)

The hive-mount post is a 15 × 15 cm square prism whose cross-section sits
in the **xz-plane** (centered on `hive_x` and `hive_z`) and whose length
runs along **y**, from `hive_y - POST_Y_BELOW_M` up to the top of the
camera frame. A visit is `post`-rejected when:

```
|x - hive_x| < POST_HALF_WIDTH_M  AND  |z - hive_z| < POST_HALF_WIDTH_M
AND  hive_y - POST_Y_BELOW_M  ≤  y  ≤  y_top_of_frame
```

(The earlier draft mistakenly used `|y - hive_y|` instead of `|z - hive_z|`
for the cross-section, which only caught bees exactly at the hive's height
rather than bees at any height behind the post.)


In [ ]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import re

import numpy as np
import pandas as pd

# ── detector knobs ──
HIVE_RADIUS_M    = 0.15
DISP_WINDOW_S    = 1.0
DISP_THR_M       = 0.05
MIN_VISIT_S      = 1.0
LINK_DIST_M      = 0.10
LINK_DT_MAX_S    = 30.0
CLUSTER_RADIUS_M = 0.15

# ── spatial filters ──
POST_HALF_WIDTH_M = 0.075   # 15 × 15 cm cross-section centred on hive (x, z)
POST_Y_BELOW_M    = 0.15    # post extends in y from hive_y - this, up to top of frame
EDGE_BUFFER_M     = 0.15    # buffer from per-axis 1st/99th percentile

# ── deceleration check (cross-track only) ──
VEL_WINDOW          = 10    # number of trailing frames considered for track A
DECEL_END_THR_MS    = 0.5   # mean speed over last half of window must be below this
DECEL_SLOPE_THR_MS  = 0.0   # slope of speed vs time (m/s²) must be <= this (i.e. decreasing or flat-low)
DECEL_DROP_RATIO    = 0.7   # second-half mean speed must be <= this × first-half mean (i.e. ≥30% drop)

# ── hive positions ──
HIVE_XYZ = {900: (-0.040, -0.665, -1.195),
            939: (-0.086, -0.828, -1.045)}

DATA_BASE = Path("../../../data/flight_data")
OUT_DIR   = Path("data/multi_day")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FOLDER_RE = re.compile(r"^(\d{4}-\d{2}-\d{2})_system_(\d+)$")

print(f"Source: {DATA_BASE.resolve()}")
print(f"Output: {OUT_DIR.resolve()}")
print(f"Post:   {2*POST_HALF_WIDTH_M*100:.0f} × {2*POST_HALF_WIDTH_M*100:.0f} cm square")
print(f"FOV edge buffer: {EDGE_BUFFER_M*100:.0f} cm")
print(f"Cross-track deceleration: end-mean ≤ {DECEL_END_THR_MS} m/s, "
      f"slope ≤ {DECEL_SLOPE_THR_MS} m/s², drop-ratio ≤ {DECEL_DROP_RATIO}")


## 1. Detector + deceleration-check functions

`track_endpoint_kinematics` is the new, more thorough velocity helper.
For every track it returns

- `end_mean_v` — mean speed over the *last half* of the last `VEL_WINDOW`
  valid frames (the bee's terminal speed, ignoring the noisy outer frames)
- `end_slope_v` — least-squares slope of speed vs time over the same
  window (negative ⇒ decelerating)
- `end_first_half_v`, `end_second_half_v` — means used for the drop-ratio
  check (cheap robustness against a single very low last frame)
- `start_mean_v` — mean speed over the first `VEL_WINDOW` valid frames of
  the track (used only for diagnostics, not gating).

`is_decelerating` then encodes the gate: the last-half mean must be
below `DECEL_END_THR_MS`, **and** either the slope is negative or the
second-half mean is `≤ DECEL_DROP_RATIO ×` first-half mean.


In [ ]:
def per_track_stationary_segments(ft_sub, disp_window_s, disp_thr, min_visit_s):
    """Scenario 1: within-track stationary segments (bee parked on a flower)."""
    if len(ft_sub) < 30: return []
    times = ft_sub["elapsed"].values
    xs = ft_sub["sposX_insect"].values
    ys = ft_sub["sposY_insect"].values
    zs = ft_sub["sposZ_insect"].values
    disp = np.full(len(ft_sub), np.nan)
    j = 0
    for i in range(len(ft_sub)):
        while j < i and times[i] - times[j] >= disp_window_s:
            j += 1
        ref = max(0, j - 1)
        if i - ref >= 5:
            disp[i] = np.sqrt((xs[i]-xs[ref])**2 + (ys[i]-ys[ref])**2 + (zs[i]-zs[ref])**2)
    stat = (disp < disp_thr) & ~np.isnan(disp)

    out, in_run, rs = [], False, 0
    for i, s in enumerate(stat):
        if s and not in_run: rs = i; in_run = True
        elif not s and in_run:
            t0, t1 = times[rs], times[i-1]
            if t1 - t0 >= min_visit_s:
                out.append((t0, t1, xs[rs:i].mean(), ys[rs:i].mean(), zs[rs:i].mean()))
            in_run = False
    if in_run:
        t0, t1 = times[rs], times[-1]
        if t1 - t0 >= min_visit_s:
            out.append((t0, t1, xs[rs:].mean(), ys[rs:].mean(), zs[rs:].mean()))
    return out


def track_endpoint_kinematics(ft, n_window):
    """For every detection_uid, return a dict of velocity descriptors on
    the last `n_window` valid frames (and the first `n_window` valid frames
    for a start_mean_v reference).

    Returns
    -------
    dict[uid] -> dict with keys:
      end_mean_v       (m/s) — mean speed over last half of trailing window
      end_full_mean_v  (m/s) — mean speed over the full trailing window
      end_slope_v      (m/s²) — slope of speed vs time over trailing window
      end_first_half_v, end_second_half_v (m/s)
      start_mean_v     (m/s) — mean speed over leading window (diagnostic)
    """
    out = {}
    for uid, sub in ft.groupby("detection_uid"):
        sub = sub.sort_values("elapsed")
        valid = sub[sub["vel_valid_insect"] == 1] if "vel_valid_insect" in sub.columns else sub
        rec = {"end_mean_v": np.nan, "end_full_mean_v": np.nan,
               "end_slope_v": np.nan, "end_first_half_v": np.nan,
               "end_second_half_v": np.nan, "start_mean_v": np.nan}
        if len(valid) >= 4:
            tail = valid.tail(n_window)
            t = tail["elapsed"].values.astype(float)
            sp = np.sqrt(tail["svelX_insect"]**2 + tail["svelY_insect"]**2 + tail["svelZ_insect"]**2).values
            rec["end_full_mean_v"] = float(np.mean(sp))
            half = max(2, len(sp) // 2)
            rec["end_first_half_v"]  = float(np.mean(sp[:half]))
            rec["end_second_half_v"] = float(np.mean(sp[-half:]))
            rec["end_mean_v"]        = rec["end_second_half_v"]
            if len(t) >= 3 and (t[-1] - t[0]) > 1e-6:
                # slope (m/s²): negative ⇒ decelerating
                rec["end_slope_v"] = float(np.polyfit(t - t[0], sp, 1)[0])
            head = valid.head(n_window)
            sp0 = np.sqrt(head["svelX_insect"]**2 + head["svelY_insect"]**2 + head["svelZ_insect"]**2).values
            rec["start_mean_v"] = float(np.mean(sp0))
        out[uid] = rec
    return out


def is_decelerating(kin, end_thr, slope_thr, drop_ratio):
    """True if the bee was decelerating to land on a flower at the end of track A.

    Logic:
      * end_second_half_v (≈ terminal speed) must be < end_thr
      * AND (slope < slope_thr) OR (second_half ≤ drop_ratio × first_half)
        — at least one of these has to indicate the bee was slowing down
        rather than just consistently moving at a low cruise speed.

    Either condition alone would let a slow but steady cruiser through.
    """
    if not np.isfinite(kin["end_mean_v"]):
        return False, "no_kinematics"
    if kin["end_mean_v"] >= end_thr:
        return False, "end_v_too_high"
    decel_slope = np.isfinite(kin["end_slope_v"]) and (kin["end_slope_v"] < slope_thr)
    decel_drop  = (np.isfinite(kin["end_first_half_v"]) and kin["end_first_half_v"] > 1e-3
                   and (kin["end_second_half_v"] <= drop_ratio * kin["end_first_half_v"]))
    if not (decel_slope or decel_drop):
        return False, "no_deceleration"
    return True, "ok"


def track_endpoints(ft):
    g = ft.sort_values("elapsed").groupby("detection_uid")
    first = g.first(); last = g.last()
    return pd.DataFrame({
        "uid": first.index,
        "t0": first["elapsed"].values, "t1": last["elapsed"].values,
        "x0": first["sposX_insect"].values, "y0": first["sposY_insect"].values, "z0": first["sposZ_insect"].values,
        "x1": last["sposX_insect"].values,  "y1": last["sposY_insect"].values,  "z1": last["sposZ_insect"].values,
        "n_frames": g.size().values,
    }).reset_index(drop=True)


def detect_visits_one_day(ft, hive_xyz):
    """Run scenario-1 (within-track) and scenario-2 (cross-track) detectors.
    Cross-track candidates pass through `is_decelerating` on track A."""
    kin_by_uid = track_endpoint_kinematics(ft, VEL_WINDOW)

    # — Scenario 1: within-track stationary segments —
    within = []
    for uid, sub in ft.sort_values("elapsed").groupby("detection_uid"):
        for t0, t1, x, y, z in per_track_stationary_segments(
                sub, DISP_WINDOW_S, DISP_THR_M, MIN_VISIT_S):
            within.append({
                "source": "within", "t_start": t0, "t_end": t1,
                "x": x, "y": y, "z": z, "duration_s": t1-t0,
                "uids": [int(uid)],
                "end_v_a": np.nan, "end_slope_a": np.nan,
                "end_first_half_v_a": np.nan, "end_second_half_v_a": np.nan,
                "start_v_b": np.nan,
            })

    # — Scenario 2: cross-track pair with deceleration gate —
    endpoints = track_endpoints(ft).sort_values("t1").reset_index(drop=True)
    cross = []
    rejected_decel = []
    for _, a in endpoints.iterrows():
        kin_a = kin_by_uid.get(a["uid"], None)
        if kin_a is None: continue
        cand = endpoints[(endpoints["t0"] > a["t1"] + MIN_VISIT_S) &
                          (endpoints["t0"] < a["t1"] + LINK_DT_MAX_S) &
                          (endpoints["uid"] != a["uid"])]
        if cand.empty: continue
        d = np.sqrt((cand["x0"]-a["x1"])**2 + (cand["y0"]-a["y1"])**2 + (cand["z0"]-a["z1"])**2)
        cand = cand.assign(d=d.values)
        cand = cand[cand["d"] < LINK_DIST_M]
        if cand.empty: continue
        b = cand.sort_values("t0").iloc[0]
        kin_b = kin_by_uid.get(int(b["uid"]), None)
        start_v_b = (kin_b["start_mean_v"] if kin_b is not None else np.nan)

        rec = {"source": "cross", "t_start": a["t1"], "t_end": b["t0"],
               "x": 0.5*(a["x1"]+b["x0"]), "y": 0.5*(a["y1"]+b["y0"]), "z": 0.5*(a["z1"]+b["z0"]),
               "duration_s": b["t0"]-a["t1"],
               "uids": [int(a["uid"]), int(b["uid"])],
               "end_v_a":           kin_a["end_mean_v"],
               "end_slope_a":       kin_a["end_slope_v"],
               "end_first_half_v_a":  kin_a["end_first_half_v"],
               "end_second_half_v_a": kin_a["end_second_half_v"],
               "start_v_b":         start_v_b}

        ok, why = is_decelerating(kin_a, DECEL_END_THR_MS, DECEL_SLOPE_THR_MS, DECEL_DROP_RATIO)
        if not ok:
            rec["filter_reason"] = "end_velocity"
            rec["decel_fail_reason"] = why
            rejected_decel.append(rec)
        else:
            cross.append(rec)

    visits = within + cross
    cols = ["source","t_start","t_end","x","y","z","duration_s","uids",
            "end_v_a","end_slope_a","end_first_half_v_a","end_second_half_v_a",
            "start_v_b","dist_hive_m","filter_reason"]
    df     = pd.DataFrame(visits) if visits else pd.DataFrame(columns=cols)
    rej_df = pd.DataFrame(rejected_decel)

    hx, hy, hz = hive_xyz
    if not df.empty:
        df["dist_hive_m"] = np.sqrt((df["x"]-hx)**2 + (df["y"]-hy)**2 + (df["z"]-hz)**2)
    if not rej_df.empty:
        rej_df["dist_hive_m"] = np.sqrt((rej_df["x"]-hx)**2 + (rej_df["y"]-hy)**2 + (rej_df["z"]-hz)**2)
    return df, rej_df


def apply_spatial_filters(df, hive_xyz, fov_bounds):
    """Mark visits with filter_reason for hive / post / fov_edge.

    POST GEOMETRY: cross-section is in the xz-plane (15x15 cm centered on
    hive_x and hive_z); length runs along y from hive_y - POST_Y_BELOW_M
    up to the top of the FOV.  A visit is post-occluded if its (x, z) lies
    inside the cross-section AND its y lies within the post's vertical
    extent.
    """
    if df.empty:
        df["filter_reason"] = None
        return df
    hx, hy, hz = hive_xyz
    x_lo, x_hi = fov_bounds["x"]; y_lo, y_hi = fov_bounds["y"]; z_lo, z_hi = fov_bounds["z"]
    reason = pd.Series([None]*len(df), index=df.index, dtype=object)

    # 1. hive (3D distance)
    d3 = np.sqrt((df["x"]-hx)**2 + (df["y"]-hy)**2 + (df["z"]-hz)**2)
    reason.loc[(reason.isna()) & (d3 < HIVE_RADIUS_M)] = "hive"

    # 2. 15x15 cm square post: cross-section in xz, length along y
    in_post_xz = (np.abs(df["x"] - hx) < POST_HALF_WIDTH_M) & \
                 (np.abs(df["z"] - hz) < POST_HALF_WIDTH_M)
    in_post_y  = (df["y"] >= hy - POST_Y_BELOW_M) & (df["y"] <= y_hi)
    in_post    = in_post_xz & in_post_y
    reason.loc[(reason.isna()) & in_post] = "post"

    # 3. FOV edges (any of 6 axis-aligned planes)
    near = ((df["x"] < x_lo + EDGE_BUFFER_M) | (df["x"] > x_hi - EDGE_BUFFER_M) |
            (df["y"] < y_lo + EDGE_BUFFER_M) | (df["y"] > y_hi - EDGE_BUFFER_M) |
            (df["z"] < z_lo + EDGE_BUFFER_M) | (df["z"] > z_hi - EDGE_BUFFER_M))
    reason.loc[(reason.isna()) & near] = "fov_edge"

    df["filter_reason"] = reason
    return df

def cluster_into_flowers(visits, radius=CLUSTER_RADIUS_M):
    flower_id = np.full(len(visits), -1, dtype=int)
    next_id = 0
    for i in range(len(visits)):
        if flower_id[i] != -1: continue
        flower_id[i] = next_id
        queue = [i]
        while queue:
            k = queue.pop()
            for j in range(len(visits)):
                if flower_id[j] != -1: continue
                d = np.sqrt((visits.iloc[k]["x"]-visits.iloc[j]["x"])**2 +
                            (visits.iloc[k]["y"]-visits.iloc[j]["y"])**2 +
                            (visits.iloc[k]["z"]-visits.iloc[j]["z"])**2)
                if d < radius:
                    flower_id[j] = next_id; queue.append(j)
        next_id += 1
    return flower_id


## 2. Compute FOV bounds per system from valid frames

The 1st and 99th percentile of all valid bee positions across the
experimental window give an empirical tracking-volume boundary per
system.  Visits within `EDGE_BUFFER_M` (= 15 cm) of any of those bounds
get the `fov_edge` filter reason.


In [ ]:
SAMPLE_DAYS_PER_SYS = {900: ["2026-04-18", "2026-04-27", "2026-05-02"],
                       939: ["2026-04-23", "2026-05-02", "2026-05-08"]}

fov_bounds = {}
for sid, days in SAMPLE_DAYS_PER_SYS.items():
    xs, ys, zs = [], [], []
    for d in days:
        folder = DATA_BASE / f"{d}_system_{sid}"
        ft_csv = folder / "flight_tracks.csv"
        if not ft_csv.exists(): continue
        ft = pd.read_csv(ft_csv, usecols=["pos_valid_insect",
                                           "sposX_insect", "sposY_insect", "sposZ_insect"])
        ft = ft[ft["pos_valid_insect"] == 1]
        xs.append(ft["sposX_insect"]); ys.append(ft["sposY_insect"]); zs.append(ft["sposZ_insect"])
    x = pd.concat(xs); y = pd.concat(ys); z = pd.concat(zs)
    fov_bounds[sid] = {
        "x": (float(np.percentile(x, 1)), float(np.percentile(x, 99))),
        "y": (float(np.percentile(y, 1)), float(np.percentile(y, 99))),
        "z": (float(np.percentile(z, 1)), float(np.percentile(z, 99))),
    }
    print(f"sys {sid} FOV bounds (1-99 pct from {len(x):,} valid frames):")
    print(f"  x: {fov_bounds[sid]['x'][0]:+.3f} -> {fov_bounds[sid]['x'][1]:+.3f}")
    print(f"  y: {fov_bounds[sid]['y'][0]:+.3f} -> {fov_bounds[sid]['y'][1]:+.3f}")
    print(f"  z: {fov_bounds[sid]['z'][0]:+.3f} -> {fov_bounds[sid]['z'][1]:+.3f}")


## 3. Sweep all day-system folders

In [ ]:
all_kept_rows = []
all_rejected_rows = []
summary_rows = []

folders = sorted(DATA_BASE.glob("*_system_*"))
print(f"Sweeping {len(folders)} folders ...")
for folder in folders:
    m = FOLDER_RE.match(folder.name)
    if not m: continue
    date_str, sys_str = m.groups()
    sid = int(sys_str)
    if sid not in HIVE_XYZ:
        print(f"  skip {folder.name}: unknown system"); continue
    ft_csv = folder / "flight_tracks.csv"
    if not ft_csv.exists():
        print(f"  skip {folder.name}: no flight_tracks.csv"); continue

    ft = pd.read_csv(ft_csv, usecols=["detection_uid", "elapsed",
                                       "sposX_insect", "sposY_insect", "sposZ_insect",
                                       "svelX_insect", "svelY_insect", "svelZ_insect",
                                       "vel_valid_insect"])

    cand_df, rej_endv_df = detect_visits_one_day(ft, HIVE_XYZ[sid])
    cand_df = apply_spatial_filters(cand_df, HIVE_XYZ[sid], fov_bounds[sid])

    kept = cand_df[cand_df["filter_reason"].isna()].copy()
    rejected = pd.concat([cand_df[cand_df["filter_reason"].notna()], rej_endv_df], ignore_index=True)

    n_within_kept = (kept["source"] == "within").sum() if not kept.empty else 0
    n_cross_kept  = (kept["source"] == "cross").sum() if not kept.empty else 0
    rej_summary = ', '.join(f'{r}={n}' for r, n in rejected["filter_reason"].value_counts().items()) if len(rejected) else ""
    print(f"  {date_str}  sys {sid}:  "
          f"kept={len(kept):3d} (within={n_within_kept}, cross={n_cross_kept})  "
          f"rejected={len(rejected):3d}  ({rej_summary})")

    if not kept.empty:
        flower_id = cluster_into_flowers(kept)
        kept = kept.assign(flower_id=flower_id)

        for _, v in kept.iterrows():
            all_kept_rows.append({
                "date": date_str, "system_id": sid,
                "t_start_s": float(v["t_start"]), "t_end_s": float(v["t_end"]),
                "duration_s": float(v["duration_s"]),
                "x": float(v["x"]), "y": float(v["y"]), "z": float(v["z"]),
                "dist_hive_m": float(v["dist_hive_m"]),
                "end_v_a":          float(v["end_v_a"])          if pd.notna(v["end_v_a"])          else np.nan,
                "end_slope_a":      float(v["end_slope_a"])      if pd.notna(v["end_slope_a"])      else np.nan,
                "end_first_half_v_a":  float(v["end_first_half_v_a"])  if pd.notna(v["end_first_half_v_a"])  else np.nan,
                "end_second_half_v_a": float(v["end_second_half_v_a"]) if pd.notna(v["end_second_half_v_a"]) else np.nan,
                "start_v_b":        float(v["start_v_b"])        if pd.notna(v["start_v_b"])        else np.nan,
                "source": v["source"],
                "uids": ";".join(str(u) for u in v["uids"]),
                "flower_id": int(v["flower_id"]),
            })

        n_visits  = len(kept)
        n_flowers = kept["flower_id"].nunique()
        summary_rows.append({
            "date": date_str, "system_id": sid,
            "n_visits": n_visits,
            "mean_handling_time_s":   float(kept["duration_s"].mean()),
            "median_handling_time_s": float(kept["duration_s"].median()),
            "total_visit_time_s":     float(kept["duration_s"].sum()),
            "n_distinct_flowers":     int(n_flowers),
            "revisit_rate":           n_visits / n_flowers if n_flowers else np.nan,
        })
    else:
        summary_rows.append({
            "date": date_str, "system_id": sid,
            "n_visits": 0, "mean_handling_time_s": np.nan,
            "median_handling_time_s": np.nan, "total_visit_time_s": 0.0,
            "n_distinct_flowers": 0, "revisit_rate": np.nan,
        })

    for _, v in rejected.iterrows():
        all_rejected_rows.append({
            "date": date_str, "system_id": sid,
            "t_start_s": float(v["t_start"]), "t_end_s": float(v["t_end"]),
            "duration_s": float(v["duration_s"]),
            "x": float(v["x"]), "y": float(v["y"]), "z": float(v["z"]),
            "end_v_a":          float(v["end_v_a"])          if pd.notna(v.get("end_v_a"))          else np.nan,
            "end_slope_a":      float(v["end_slope_a"])      if pd.notna(v.get("end_slope_a"))      else np.nan,
            "end_first_half_v_a":  float(v["end_first_half_v_a"])  if pd.notna(v.get("end_first_half_v_a"))  else np.nan,
            "end_second_half_v_a": float(v["end_second_half_v_a"]) if pd.notna(v.get("end_second_half_v_a")) else np.nan,
            "source": v["source"],
            "uids": ";".join(str(u) for u in v["uids"]),
            "filter_reason":    v["filter_reason"],
            "decel_fail_reason": v.get("decel_fail_reason", None),
        })

print(f"\nDone.  Total kept: {len(all_kept_rows):,}  Total rejected: {len(all_rejected_rows):,}")


## 4. Write outputs

In [ ]:
kept_df     = pd.DataFrame(all_kept_rows)
rejected_df = pd.DataFrame(all_rejected_rows)
summary_df  = (pd.DataFrame(summary_rows)
                 .sort_values(["system_id", "date"])
                 .reset_index(drop=True))

kept_df.to_csv(OUT_DIR / "flower_visits.csv", index=False)
rejected_df.to_csv(OUT_DIR / "flower_visits_rejected.csv", index=False)
summary_df.to_csv(OUT_DIR / "flower_visit_summary.csv", index=False)

print(f"Wrote:")
print(f"  {OUT_DIR/'flower_visits.csv'}            ({len(kept_df):,} kept visits)")
print(f"  {OUT_DIR/'flower_visits_rejected.csv'}   ({len(rejected_df):,} rejected, with filter_reason)")
print(f"  {OUT_DIR/'flower_visit_summary.csv'}     ({len(summary_df):,} day-system rows)")

print()
print("Rejection breakdown:")
if len(rejected_df):
    print(rejected_df["filter_reason"].value_counts().to_string())
    end_velocity_rej = rejected_df[rejected_df["filter_reason"] == "end_velocity"]
    if "decel_fail_reason" in end_velocity_rej.columns and end_velocity_rej["decel_fail_reason"].notna().any():
        print("\nWhich deceleration sub-check fired (end_velocity rows only):")
        print(end_velocity_rej["decel_fail_reason"].value_counts().to_string())
else:
    print("  (no rejections)")
print()
print("Summary preview:")
print(summary_df.to_string(index=False))


## 5. Sanity plot: kept vs rejected — two projections

Top row: **top-down (xy) view**. The post appears as a vertical band of
width 15 cm along x, extending in y from `hive_y - POST_Y_BELOW_M` up to
the top of frame.

Bottom row: **cross-section (xz) view**. The post here is a 15 × 15 cm
square centered on `(hive_x, hive_z)`.

Green points (kept) should sit clear of the dashed post outlines and the
orange FOV-boundary dotted lines in *both* projections.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
plt.rcParams.update({"figure.dpi": 110, "font.size": 11})

REASON_COLOUR = {"hive": "#888888", "post": "#9333EA",
                 "fov_edge": "#F97316", "end_velocity": "#EF4444",
                 "deceleration": "#EF4444"}

systems = sorted(set(kept_df["system_id"].unique()) | set(rejected_df["system_id"].unique()))
fig, axes = plt.subplots(2, len(systems), figsize=(7 * len(systems), 12), squeeze=False)

for col, sid in enumerate(systems):
    hx, hy, hz = HIVE_XYZ[sid]
    fb = fov_bounds[sid]
    y_top = fb["y"][1]

    sub_k = kept_df[kept_df["system_id"] == sid]
    sub_r = rejected_df[rejected_df["system_id"] == sid]

    # ── top row: top-down xy view
    # the post in this projection is a vertical band of width 2*POST_HALF_WIDTH_M
    # spanning y from hy - POST_Y_BELOW_M up to y_top
    ax = axes[0][col]
    ax.scatter(sub_k["x"], sub_k["y"], s=30, c="#22C55E", alpha=0.7,
               edgecolor="black", linewidth=0.3, label=f"kept (n={len(sub_k)})")
    for reason, c in REASON_COLOUR.items():
        s = sub_r[sub_r["filter_reason"] == reason]
        if len(s):
            ax.scatter(s["x"], s["y"], s=22, c=c, alpha=0.5,
                       label=f"{reason} (n={len(s)})")
    ax.scatter([hx], [hy], marker="*", s=300, c="black", edgecolor="white",
               linewidth=0.8, label="hive", zorder=10)
    post_xy = mpatches.Rectangle(
        (hx - POST_HALF_WIDTH_M, hy - POST_Y_BELOW_M),
        2*POST_HALF_WIDTH_M, y_top - (hy - POST_Y_BELOW_M),
        fill=False, edgecolor="#9333EA", linewidth=1.5, linestyle="--",
        label="post (xy view)")
    ax.add_patch(post_xy)
    ax.axvline(fb["x"][0] + EDGE_BUFFER_M, color="#F97316", linestyle=":", linewidth=1)
    ax.axvline(fb["x"][1] - EDGE_BUFFER_M, color="#F97316", linestyle=":", linewidth=1)
    ax.axhline(fb["y"][0] + EDGE_BUFFER_M, color="#F97316", linestyle=":", linewidth=1)
    ax.axhline(fb["y"][1] - EDGE_BUFFER_M, color="#F97316", linestyle=":", linewidth=1)
    ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
    ax.set_title(f"system {sid} — top-down (xy)")
    ax.legend(fontsize=8, loc="best")
    ax.set_aspect("equal", adjustable="datalim")

    # ── bottom row: xz cross-section view
    # the post here is a 15x15 cm square centered on (hx, hz)
    ax = axes[1][col]
    ax.scatter(sub_k["x"], sub_k["z"], s=30, c="#22C55E", alpha=0.7,
               edgecolor="black", linewidth=0.3, label=f"kept (n={len(sub_k)})")
    for reason, c in REASON_COLOUR.items():
        s = sub_r[sub_r["filter_reason"] == reason]
        if len(s):
            ax.scatter(s["x"], s["z"], s=22, c=c, alpha=0.5,
                       label=f"{reason} (n={len(s)})")
    ax.scatter([hx], [hz], marker="*", s=300, c="black", edgecolor="white",
               linewidth=0.8, label="hive", zorder=10)
    post_xz = mpatches.Rectangle(
        (hx - POST_HALF_WIDTH_M, hz - POST_HALF_WIDTH_M),
        2*POST_HALF_WIDTH_M, 2*POST_HALF_WIDTH_M,
        fill=False, edgecolor="#9333EA", linewidth=1.5, linestyle="--",
        label="post cross-section (xz, 15 cm)")
    ax.add_patch(post_xz)
    ax.axvline(fb["x"][0] + EDGE_BUFFER_M, color="#F97316", linestyle=":", linewidth=1)
    ax.axvline(fb["x"][1] - EDGE_BUFFER_M, color="#F97316", linestyle=":", linewidth=1)
    ax.axhline(fb["z"][0] + EDGE_BUFFER_M, color="#F97316", linestyle=":", linewidth=1)
    ax.axhline(fb["z"][1] - EDGE_BUFFER_M, color="#F97316", linestyle=":", linewidth=1)
    ax.set_xlabel("x [m]"); ax.set_ylabel("z [m]")
    ax.set_title(f"system {sid} — cross-section (xz)")
    ax.legend(fontsize=8, loc="best")
    ax.set_aspect("equal", adjustable="datalim")

plt.tight_layout(); plt.show()

## 6. Deceleration diagnostic

Three panels for the cross-track candidates:

1. **End-mean speed** — kept (passed deceleration gate) vs rejected.
2. **End slope of speed** — should skew negative for kept, near-zero/positive for rejected.
3. **Drop-ratio (second_half / first_half)** — kept should cluster
   below `DECEL_DROP_RATIO`; rejected near 1.0 (steady speed).


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))

kept_cross = kept_df[kept_df["source"] == "cross"]
rej_endv   = rejected_df[rejected_df["filter_reason"] == "end_velocity"]

# 1) end speed
ax = axes[0]
kv = kept_cross["end_v_a"].dropna()
rv = rej_endv["end_v_a"].dropna()
hi = max(2.5, rv.max() if len(rv) else 2.5)
bins = np.linspace(0, hi, 30)
ax.hist(kv, bins=bins, alpha=0.7, color="#22C55E",
        label=f"kept  (n={len(kv)})", edgecolor="black", linewidth=0.4)
ax.hist(rv, bins=bins, alpha=0.7, color="#EF4444",
        label=f"rejected  (n={len(rv)})", edgecolor="black", linewidth=0.4)
ax.axvline(DECEL_END_THR_MS, color="black", linestyle="--", linewidth=1,
           label=f"threshold {DECEL_END_THR_MS} m/s")
ax.set_xlabel("end mean speed (last half of window) [m/s]")
ax.set_ylabel("count"); ax.set_title("Track A end-speed"); ax.legend(fontsize=8)

# 2) end slope
ax = axes[1]
ks = kept_cross["end_slope_a"].dropna()
rs = rej_endv["end_slope_a"].dropna()
edge = max(abs(ks).max() if len(ks) else 1, abs(rs).max() if len(rs) else 1, 1.0)
bins = np.linspace(-edge, edge, 40)
ax.hist(ks, bins=bins, alpha=0.7, color="#22C55E",
        label=f"kept  (n={len(ks)})", edgecolor="black", linewidth=0.4)
ax.hist(rs, bins=bins, alpha=0.7, color="#EF4444",
        label=f"rejected  (n={len(rs)})", edgecolor="black", linewidth=0.4)
ax.axvline(0.0, color="black", linestyle="--", linewidth=1, label="slope = 0")
ax.set_xlabel("slope of speed vs t [m/s²]")
ax.set_ylabel("count"); ax.set_title("Track A end-slope"); ax.legend(fontsize=8)

# 3) drop ratio
ax = axes[2]
def _ratio(df):
    a = df["end_first_half_v_a"]; b = df["end_second_half_v_a"]
    mask = (a > 1e-3) & a.notna() & b.notna()
    return (b[mask] / a[mask])
kr = _ratio(kept_cross); rr = _ratio(rej_endv)
bins = np.linspace(0, 2.5, 30)
ax.hist(kr, bins=bins, alpha=0.7, color="#22C55E",
        label=f"kept  (n={len(kr)})", edgecolor="black", linewidth=0.4)
ax.hist(rr, bins=bins, alpha=0.7, color="#EF4444",
        label=f"rejected  (n={len(rr)})", edgecolor="black", linewidth=0.4)
ax.axvline(DECEL_DROP_RATIO, color="black", linestyle="--", linewidth=1,
           label=f"drop ratio {DECEL_DROP_RATIO}")
ax.set_xlabel("second-half mean / first-half mean speed")
ax.set_ylabel("count"); ax.set_title("Track A drop-ratio"); ax.legend(fontsize=8)

plt.tight_layout(); plt.show()

print("Cross-track deceleration stats:")
if len(kv):
    print(f"  kept end speed:     median={kv.median():.3f}, max={kv.max():.3f}")
if len(rv):
    print(f"  rejected end speed: median={rv.median():.3f}, max={rv.max():.3f}")
if "decel_fail_reason" in rej_endv.columns:
    sub_reasons = rej_endv["decel_fail_reason"].value_counts()
    if len(sub_reasons):
        print("\nWhy each rejected cross-track failed:")
        print(sub_reasons.to_string())


## 7. Per-track velocity example: kept vs rejected

For a quick visual sanity-check, plot the trailing speed-vs-time profile
of one kept and one rejected cross-track candidate.  A well-decelerating
landing should show speed dropping toward the end; a tracking-dropout
artefact should show roughly constant speed.


In [ ]:
def _trailing_profile(date, sid, uid, n=VEL_WINDOW):
    folder = DATA_BASE / f"{date}_system_{sid}"
    ft_csv = folder / "flight_tracks.csv"
    if not ft_csv.exists(): return None
    ft = pd.read_csv(ft_csv, usecols=["detection_uid","elapsed",
                                       "svelX_insect","svelY_insect","svelZ_insect",
                                       "vel_valid_insect"])
    sub = ft[ft["detection_uid"] == int(uid)].sort_values("elapsed")
    sub = sub[sub["vel_valid_insect"] == 1]
    if len(sub) < 4: return None
    tail = sub.tail(n)
    t = tail["elapsed"].values - tail["elapsed"].values[0]
    sp = np.sqrt(tail["svelX_insect"]**2 + tail["svelY_insect"]**2 + tail["svelZ_insect"]**2).values
    return t, sp

def _first_uid(row):
    u = row["uids"].split(";")[0]
    try: return int(u)
    except: return None

# pick one kept and one rejected cross-track (deepest median candidate)
kept_cross_sorted = kept_df[kept_df["source"] == "cross"].sort_values("end_v_a")
rej_endv_sorted   = rejected_df[(rejected_df["filter_reason"] == "end_velocity")].sort_values("end_v_a", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
for ax, label, df_sorted, color in [
        (axes[0], "kept (decelerating)", kept_cross_sorted, "#22C55E"),
        (axes[1], "rejected (no decel)", rej_endv_sorted,   "#EF4444")]:
    plotted = 0
    for _, r in df_sorted.head(6).iterrows():
        uid = _first_uid(r)
        if uid is None: continue
        prof = _trailing_profile(r["date"], r["system_id"], uid)
        if prof is None: continue
        t, sp = prof
        ax.plot(t, sp, color=color, alpha=0.6, linewidth=1.4,
                label=f"{r['date']} sys{r['system_id']} uid{uid}")
        plotted += 1
        if plotted >= 4: break
    ax.axhline(DECEL_END_THR_MS, color="black", linestyle="--", linewidth=0.8)
    ax.set_title(label)
    ax.set_xlabel("time since first valid frame in window [s]")
    ax.legend(fontsize=7, loc="best")
axes[0].set_ylabel("speed [m/s]")
plt.tight_layout(); plt.show()


## 8. Condition comparison (final indicators)

Box-and-strip plots per indicator × system, BASELINE / OFF / ON.
y-axis is shared within each row (per project conventions for box plots
shown side-by-side).


In [ ]:
CYCLE_ANCHOR = pd.Timestamp("2026-04-23")
def label_date(d):
    d = pd.Timestamp(d)
    if d < CYCLE_ANCHOR: return "BASELINE"
    return "ON" if ((d - CYCLE_ANCHOR).days // 3) % 2 == 0 else "OFF"

summary_df["date_ts"]   = pd.to_datetime(summary_df["date"])
summary_df["condition"] = summary_df["date_ts"].apply(label_date)

INDICATORS = ["mean_handling_time_s", "n_distinct_flowers", "n_visits", "revisit_rate"]
ORDER = ["BASELINE", "OFF", "ON"]
COND_COLOUR = {"BASELINE": "#888888", "OFF": "#5B8FD4", "ON": "#E05252"}

systems = sorted(summary_df["system_id"].unique())
fig, axes = plt.subplots(len(INDICATORS), len(systems),
                         figsize=(5 * len(systems), 4 * len(INDICATORS)),
                         sharey="row", squeeze=False)
for row, ind in enumerate(INDICATORS):
    for col, sid in enumerate(systems):
        ax = axes[row][col]
        data, labels, colors = [], [], []
        for cond in ORDER:
            vals = summary_df.loc[(summary_df["system_id"] == sid) &
                                   (summary_df["condition"] == cond), ind].dropna()
            data.append(vals); labels.append(f"{cond}\nn={len(vals)}"); colors.append(COND_COLOUR[cond])
        bp = ax.boxplot(data, tick_labels=labels, patch_artist=True, widths=0.55)
        for patch, c in zip(bp["boxes"], colors):
            patch.set_facecolor(c); patch.set_alpha(0.6)
        for i, vals in enumerate(data):
            ax.scatter(np.full(len(vals), i+1) + np.random.uniform(-0.08, 0.08, len(vals)),
                       vals, color="black", s=22, alpha=0.85, zorder=3)
        if row == 0: ax.set_title(f"system {sid}")
        if col == 0: ax.set_ylabel(ind)
        ax.set_ylim(bottom=0)
fig.suptitle("Flower-visit indicators by condition (shared y per row, deceleration-filtered)",
             fontweight="bold", y=1.0)
plt.tight_layout(); plt.show()

print("Median values by (system, condition):")
for sid in systems:
    sub = summary_df[summary_df["system_id"] == sid]
    print(f"sys {sid}:")
    for ind in INDICATORS:
        meds = sub.groupby("condition")[ind].median().reindex(ORDER)
        ns   = sub.groupby("condition")[ind].count().reindex(ORDER)
        parts = []
        for c in ORDER:
            v = meds[c]; n = ns[c]
            parts.append(f"{c}={v:.2f}(n={int(n)})" if pd.notna(v) else f"{c}=NaN(n={int(n)})")
        print(f"  {ind:<28s}  " + "  ".join(parts))


## 9. Plug into `5g_foraging_effect_model.ipynb`

In the indicator-table cell of the model notebook, replace `median_ifi_s`
and `vertical_deviation` with the new flower-visit indicators:

```python
fv = pd.read_csv("data/multi_day/flower_visit_summary.csv")
fv["date"] = pd.to_datetime(fv["date"])
ind = ind.merge(fv[["date","system_id","mean_handling_time_s","n_distinct_flowers"]],
                on=["date","system_id"], how="left")

INDICATORS = [
    "neg_exit_count",         # foraging volume
    "neg_re_ratio",           # asymmetry of detection
    "path_tortuosity",        # navigation efficiency
    "ifi_cv",                 # foraging-rhythm regularity
    "mean_handling_time_s",   # Heinrich 1979 efficiency metric
    "n_distinct_flowers",     # flower-constancy proxy (Klein 2003)
]
```
